In [1]:
!pip install transformers
!pip install openpyxl
!pip install yacs-stubgen
!pip install pytorch-lightning torch metrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import re
from sklearn.metrics import classification_report
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
import random
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pytorch_lightning as L
from torchmetrics.classification import MulticlassF1Score
from transformers import BertModel
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

In [3]:
from yacs.config import CfgNode
cfg = CfgNode()
cfg.seed = 42

In [4]:
def seed_everything(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def main(cfg):
  seed_everything(cfg.seed)

In [49]:
train_df = pd.read_csv('train.csv')
valid_df = pd.read_csv('valid.csv')
test_df = pd.read_csv('test.csv')
train_df.shape, valid_df.shape, test_df.shape

((3399, 2), (728, 2), (729, 2))

In [50]:
train_df["label"].value_counts()

,count
label,
against,1217
neutral,1129
favor,1053


In [51]:
# Label mapping
label2id = {
    "against": 0,
    "favor": 1,
    "neutral": 2
}

id2label = {v: k for k, v in label2id.items()}

for df in [train_df, valid_df, test_df]:
    df["label"] = (
        df["label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(label2id)
        .astype(int)
    )

print(train_df["label"].value_counts().sort_index())
print(valid_df["label"].value_counts().sort_index())
print(test_df["label"].value_counts().sort_index())

label
0    1217
1    1053
2    1129
Name: count, dtype: int64
label
0    261
1    225
2    242
Name: count, dtype: int64
label
0    261
1    226
2    242
Name: count, dtype: int64


In [52]:
train_df

,text,label
0,The rise in global average temperatures stoppe...,0
1,HillaryClinton I wish u were in Benghazi inste...,0
2,Climate change is a global emergency.,1
3,Agree choice if woman's life in danger. But ra...,0
4,DeanLeh earthguardianz excellent speech! Well ...,2
...,...,...
3394,USCRN temperature stations show no warming sin...,0
3395,kimkennedy4 thats why they still have no votin...,2
3396,WORLDWIDE R.E. Agents! - Support a ProLife Gro...,0
3397,GlblWarmingNews And the tooth fairy might be c...,0


In [53]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [54]:
max_length = 256
batch_size = 32

def to_encodings(df):
  texts = df["text"].tolist()
  labels = df["label"].values
  encodings = tokenizer(
      texts,
      truncation=True,
      padding="max_length",
      max_length=max_length,
      return_tensors="pt"
  )
  return encodings

train_enc = to_encodings(train_df)
valid_enc = to_encodings(valid_df)
test_enc = to_encodings(test_df)

In [55]:
for i in range(10):
  tokens = tokenizer.convert_ids_to_tokens(train_enc["input_ids"][i])
  print("Original text:")
  print(train_df.loc[i, "text"])

  print("Tokens:")
  print(tokens)
  print('\n')

Original text:
The rise in global average temperatures stopped in the late 1990s.
Tokens:
['[CLS]', 'the', 'rise', 'in', 'global', 'average', 'temperatures', 'stopped', 'in', 'the', 'late', '1990s', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PA

In [56]:
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [57]:
labels = train_df["label"].values
val_labels = valid_df["label"].values
dataset = TextDataset(train_enc, labels)
labels.shape

(3399,)

In [58]:
train_dataset = TextDataset(train_enc, train_df.label.values)
valid_dataset = TextDataset(valid_enc, valid_df.label.values)
test_dataset = TextDataset(test_enc, test_df.label.values)

In [59]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

#model-1

In [60]:
class BertClassifier(L.LightningModule):

    def __init__(
        self,
        num_classes=3,
        lr=2e-5,
        pretrained_model="bert-base-uncased"
    ):
        super().__init__()

        self.save_hyperparameters()

        self.bert = BertModel.from_pretrained(pretrained_model)
        self.bert.resize_token_embeddings(len(tokenizer))

        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(hidden_size, num_classes)

        self.criterion = nn.CrossEntropyLoss()

        self.val_f1 = MulticlassF1Score(
            num_classes=num_classes,
            average="macro"
        )

        self.test_f1 = MulticlassF1Score(
            num_classes=num_classes,
            average="macro"
        )

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0]

        logits = self.fc(
            self.dropout(cls_output)
        )

        return logits

    def training_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.criterion(
            logits,
            batch["labels"]
        )

        self.log("train_loss", loss, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.criterion(
            logits,
            batch["labels"]
        )

        preds = torch.argmax(logits, dim=1)

        self.log("val_loss", loss, prog_bar=True)
        self.log(
            "val_macro_f1",
            self.val_f1(preds, batch["labels"]),
            prog_bar=True
        )

    def test_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.criterion(
            logits,
            batch["labels"]
        )

        preds = torch.argmax(logits, dim=1)

        self.log("test_loss", loss, prog_bar=True)
        self.log(
            "test_macro_f1",
            self.test_f1(preds, batch["labels"]),
            prog_bar=True
        )

    def configure_optimizers(self):

        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr
        )

        return optimizer

In [61]:
model = BertClassifier(
    num_classes=3,
    lr=2e-5,
    pretrained_model="bert-base-uncased"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [62]:
early_stopping = EarlyStopping(
    monitor="val_macro_f1",
    mode="max",
    patience=3
)

checkpoint = ModelCheckpoint(
    monitor="val_macro_f1",
    mode="max",
    save_top_k=1,
    filename="bert-best"
)

In [63]:
trainer = L.Trainer(
    max_epochs=5,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    callbacks=[
        early_stopping,
        checkpoint
    ],
    log_every_n_steps=10
)

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [64]:
trainer.fit(
    model,
    train_loader,
    valid_loader
)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ bert      │ BertModel         │  109 M │ eval  │     0 │
│ 1 │ dropout   │ Dropout           │      0 │ train │     0 │
│ 2 │ fc        │ Linear            │  2.3 K │ train │     0 │
│ 3 │ criterion │ CrossEntropyLoss  │      0 │ train │     0 │
│ 4 │ val_f1    │ MulticlassF1Score │      0 │ train │     0 │
│ 5 │ test_f1   │ MulticlassF1Score │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 109 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 109 M                                                                                                
Total estimated model params size (MB): 437.938                                                                    
Modules in train mode: 5                                                                                           
Modules in eval mode: 228                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:538: Found 228 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [65]:
best_model = BertClassifier.load_from_checkpoint(
    checkpoint.best_model_path
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [66]:
trainer.test(
    best_model,
    dataloaders=test_loader
)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    1.3188899755477905     │
│       test_macro_f1       │    0.6418144106864929     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.3188899755477905, 'test_macro_f1': 0.6418144106864929}]